# 04 — Modeling Template for RQ1–RQ5

This notebook is a starter template only.
It gives you one clean, reproducible structure for:
- RQ1 regression
- RQ2 feature significance / importance
- RQ3 anomaly detection
- RQ4 classification
- RQ5 model comparison

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, f1_score, roc_auc_score
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, IsolationForest
from sklearn.linear_model import LinearRegression, LogisticRegression
import numpy as np

project_root = Path("..").resolve()
df = pd.read_parquet(project_root / "data" / "processed" / "final_dataset.parquet")
df = df.sort_values(["usgs_site_no", "date"]).copy()
df.head()

In [ ]:
# Example target setup for RQ1: next-day turbidity prediction
if "turbidity" in df.columns:
    df["turbidity_t_plus_1"] = df.groupby("usgs_site_no")["turbidity"].shift(-1)

feature_cols = [
    c for c in df.columns
    if c not in ["site_id", "usgs_site_no", "date", "turbidity_t_plus_1", "compliance_flag"]
    and pd.api.types.is_numeric_dtype(df[c])
]

model_df = df.dropna(subset=["turbidity_t_plus_1"] + feature_cols).copy()
X = model_df[feature_cols]
y = model_df["turbidity_t_plus_1"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

lr = LinearRegression()
rf = RandomForestRegressor(n_estimators=200, random_state=42)

lr.fit(X_train, y_train)
rf.fit(X_train, y_train)

for name, model in [("LinearRegression", lr), ("RandomForest", rf)]:
    pred = model.predict(X_test)
    rmse = mean_squared_error(y_test, pred, squared=False)
    mae = mean_absolute_error(y_test, pred)
    r2 = r2_score(y_test, pred)
    print(name, {"RMSE": rmse, "MAE": mae, "R2": r2})

In [ ]:
# Example RQ4: compliance classification
clf_df = df.dropna(subset=["compliance_flag"]).copy()
clf_feature_cols = [c for c in feature_cols if c in clf_df.columns]
clf_df = clf_df.dropna(subset=clf_feature_cols)

if len(clf_df) > 0 and clf_df["compliance_flag"].nunique() > 1:
    Xc = clf_df[clf_feature_cols]
    yc = clf_df["compliance_flag"].astype(int)

    Xc_train, Xc_test, yc_train, yc_test = train_test_split(
        Xc, yc, test_size=0.2, random_state=42, stratify=yc
    )

    logit = LogisticRegression(max_iter=2000)
    rf_clf = RandomForestClassifier(n_estimators=200, random_state=42)

    logit.fit(Xc_train, yc_train)
    rf_clf.fit(Xc_train, yc_train)

    for name, model in [("LogisticRegression", logit), ("RandomForestClassifier", rf_clf)]:
        pred = model.predict(Xc_test)
        proba = model.predict_proba(Xc_test)[:, 1]
        print(name, {
            "F1": f1_score(yc_test, pred),
            "ROC_AUC": roc_auc_score(yc_test, proba)
        })

In [ ]:
# Example RQ3: anomaly detection
anom_features = [c for c in feature_cols if c in df.columns]
anom_df = df.dropna(subset=anom_features).copy()

if len(anom_df) > 200:
    iso = IsolationForest(contamination=0.05, random_state=42)
    anom_df["anomaly_score"] = iso.fit_predict(anom_df[anom_features])
    anom_df["anomaly_flag"] = (anom_df["anomaly_score"] == -1).astype(int)
    anom_df[["date", "usgs_site_no", "anomaly_flag"]].head()